# Pepper — Colab Training Notebook

End-to-end training on Google Colab. This notebook is the primary runner
for the GPU-bound parts of the project; the Mac side of the workflow is
scoped to code editing, raw image curation, and final report assembly.

Flow:
  0. Sanity-check the allocated GPU.
  1. Mount Google Drive.
  2. Configure Drive-backed cache paths.
  3. Clone the repo and install dependencies.
  4. One-time: download Stable Diffusion 1.5 into the Drive cache.
  5. Sync raw Pepper photos from Drive into the working tree.
  6. Preprocess + caption the dataset.
  7. Train Textual Inversion (checkpoints stream to Drive).
  8. Train DreamBooth (checkpoints stream to Drive).
  9. Evaluate one or both checkpoints with the manual DDIM sampler + CLIP.

Recommended Colab runtimes: T4 (free) is sufficient for both methods; L4
or A100 (Pro) roughly 3x faster for DreamBooth.

## 0. GPU sanity check

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), (
    'No CUDA device detected. In Colab: Runtime > Change runtime type > GPU.'
)
print('CUDA device:', torch.cuda.get_device_name(0))
print('Total VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1. Mount Google Drive

Everything that needs to survive runtime disconnects lives under Drive:
raw photos, the SD 1.5 weight cache, and training checkpoints.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Drive-backed cache layout

Edit `DRIVE_CACHE_ROOT` and `DRIVE_RAW_IMAGES_DIRECTORY` to match where you
keep Pepper's photos and where you want weights cached. Everything else is
derived from these two paths.

In [ ]:
import os
from pathlib import Path

DRIVE_CACHE_ROOT = Path('/content/drive/MyDrive/persistent-pepper-cache')
DRIVE_RAW_IMAGES_DIRECTORY = Path('/content/drive/MyDrive/persistent-pepper-data/raw')

DRIVE_SD_MODEL_DIRECTORY = DRIVE_CACHE_ROOT / 'sd-v1-5'
DRIVE_CHECKPOINTS_ROOT = DRIVE_CACHE_ROOT / 'checkpoints'
DRIVE_CLASS_PRIOR_DIRECTORY = DRIVE_CACHE_ROOT / 'class_prior_dog'
DRIVE_PROCESSED_IMAGES_DIRECTORY = DRIVE_CACHE_ROOT / 'processed'
DRIVE_CAPTIONS_FILE_PATH = DRIVE_CACHE_ROOT / 'captions.json'

for directory_to_ensure in [
    DRIVE_CACHE_ROOT,
    DRIVE_SD_MODEL_DIRECTORY,
    DRIVE_CHECKPOINTS_ROOT,
    DRIVE_CLASS_PRIOR_DIRECTORY,
    DRIVE_PROCESSED_IMAGES_DIRECTORY,
]:
    directory_to_ensure.mkdir(parents=True, exist_ok=True)

os.environ['PEPPER_SD_MODEL'] = str(DRIVE_SD_MODEL_DIRECTORY)
print('SD 1.5 cache:      ', DRIVE_SD_MODEL_DIRECTORY)
print('Checkpoints root:  ', DRIVE_CHECKPOINTS_ROOT)
print('Raw images source: ', DRIVE_RAW_IMAGES_DIRECTORY)

## 3. Clone the repository and install dependencies

Point `GITHUB_REPOSITORY_URL` at your own fork. If you prefer to keep the
repo in Drive and just symlink it, replace the `git clone` line with
`!ln -s /content/drive/MyDrive/persistent-pepper /content/persistent-pepper`.

In [ ]:
GITHUB_REPOSITORY_URL = 'https://github.com/<your-username>/persistent-pepper.git'
LOCAL_REPOSITORY_ROOT = Path('/content/persistent-pepper')

if not LOCAL_REPOSITORY_ROOT.exists():
    !git clone {GITHUB_REPOSITORY_URL} {LOCAL_REPOSITORY_ROOT}
else:
    !cd {LOCAL_REPOSITORY_ROOT} && git pull --ff-only

%cd {LOCAL_REPOSITORY_ROOT}
!pip -q install -r requirements.txt

## 4. One-time: download Stable Diffusion 1.5 into the Drive cache

Runs once per Drive account. If `DRIVE_SD_MODEL_DIRECTORY` already has the
weights, this no-ops within seconds. The first run downloads ~4 GB.

In [ ]:
weights_already_cached = (DRIVE_SD_MODEL_DIRECTORY / 'unet' / 'diffusion_pytorch_model.safetensors').is_file() \
    or (DRIVE_SD_MODEL_DIRECTORY / 'unet' / 'diffusion_pytorch_model.bin').is_file()

if not weights_already_cached:
    !huggingface-cli download \
        stable-diffusion-v1-5/stable-diffusion-v1-5 \
        --local-dir {DRIVE_SD_MODEL_DIRECTORY} \
        --local-dir-use-symlinks False
else:
    print('SD 1.5 weights already cached at', DRIVE_SD_MODEL_DIRECTORY)

## 5. Sync raw Pepper photos from Drive to the local working tree

We copy rather than symlink so the preprocess step reads from local SSD,
which is 5-10x faster than reading from Drive on each call.

In [ ]:
LOCAL_RAW_IMAGES_DIRECTORY = LOCAL_REPOSITORY_ROOT / 'data' / 'raw'
LOCAL_RAW_IMAGES_DIRECTORY.mkdir(parents=True, exist_ok=True)

!rsync -a --delete \
    --exclude '.gitkeep' \
    {DRIVE_RAW_IMAGES_DIRECTORY}/ {LOCAL_RAW_IMAGES_DIRECTORY}/

raw_file_count = sum(
    1 for _ in LOCAL_RAW_IMAGES_DIRECTORY.iterdir() if _.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}
)
print(f'{raw_file_count} raw images present in {LOCAL_RAW_IMAGES_DIRECTORY}')
assert raw_file_count > 0, (
    f'No raw images found. Upload your Pepper photos to {DRIVE_RAW_IMAGES_DIRECTORY} first.'
)

## 6. Preprocess and caption

Processed images and the captions JSON are written directly to the Drive
cache so subsequent runs skip the BLIP caption pass (which is the slowest
part of dataset prep).

In [ ]:
!python scripts/preprocess_images.py \
    --raw_images_directory {LOCAL_RAW_IMAGES_DIRECTORY} \
    --processed_images_directory {DRIVE_PROCESSED_IMAGES_DIRECTORY} \
    --target_resolution 512

In [ ]:
if not DRIVE_CAPTIONS_FILE_PATH.is_file() or DRIVE_CAPTIONS_FILE_PATH.stat().st_size < 10:
    !python scripts/generate_captions.py \
        --processed_images_directory {DRIVE_PROCESSED_IMAGES_DIRECTORY} \
        --captions_output_path {DRIVE_CAPTIONS_FILE_PATH}
else:
    print('Captions already cached at', DRIVE_CAPTIONS_FILE_PATH)

## 7. Train Textual Inversion

Overrides redirect every input/output path at the Drive cache so the run
is crash-safe. Checkpoint cadence is controlled by
`checkpoint_every_n_steps` in the config.

In [ ]:
TEXTUAL_INVERSION_OUTPUT_DIRECTORY = DRIVE_CHECKPOINTS_ROOT / 'textual_inversion'

!python scripts/train.py \
    --method textual_inversion \
    --config configs/textual_inv_cfg.yaml \
    --override processed_images_directory={DRIVE_PROCESSED_IMAGES_DIRECTORY} \
    --override captions_file_path={DRIVE_CAPTIONS_FILE_PATH} \
    --override output_directory={TEXTUAL_INVERSION_OUTPUT_DIRECTORY}

## 8. Train DreamBooth

First run on a fresh Drive will auto-generate ~100 class-prior dog images
into `DRIVE_CLASS_PRIOR_DIRECTORY`; subsequent runs reuse them. Full
UNet checkpoints are ~3.4 GB each, so set `checkpoint_every_n_steps` in
the YAML to a value you're comfortable writing to Drive.

In [ ]:
DREAMBOOTH_OUTPUT_DIRECTORY = DRIVE_CHECKPOINTS_ROOT / 'dreambooth'

!python scripts/train.py \
    --method dreambooth \
    --config configs/dreambooth_cfg.yaml \
    --override processed_images_directory={DRIVE_PROCESSED_IMAGES_DIRECTORY} \
    --override captions_file_path={DRIVE_CAPTIONS_FILE_PATH} \
    --override class_prior_images_directory={DRIVE_CLASS_PRIOR_DIRECTORY} \
    --override output_directory={DREAMBOOTH_OUTPUT_DIRECTORY}

## 9. Evaluate

Renders images for each stress-test prompt and writes a CLIP-I / CLIP-T
report. Swap `METHOD_TO_EVALUATE` between the two runs to compare.

In [ ]:
METHOD_TO_EVALUATE = 'dreambooth'  # or 'textual_inversion'

if METHOD_TO_EVALUATE == 'dreambooth':
    checkpoint_path_for_evaluation = DREAMBOOTH_OUTPUT_DIRECTORY / 'latest'
    subject_identifier_phrase = 'sks dog'
else:
    checkpoint_path_for_evaluation = TEXTUAL_INVERSION_OUTPUT_DIRECTORY / 'learned_embedding_final.bin'
    subject_identifier_phrase = '<pepper>'

evaluation_output_directory = DRIVE_CHECKPOINTS_ROOT / f'eval_{METHOD_TO_EVALUATE}'

!python scripts/evaluate.py \
    --method {METHOD_TO_EVALUATE} \
    --pretrained_model_name_or_path {DRIVE_SD_MODEL_DIRECTORY} \
    --checkpoint_path {checkpoint_path_for_evaluation} \
    --reference_images_directory {DRIVE_PROCESSED_IMAGES_DIRECTORY} \
    --prompts_file configs/stress_test_prompts.txt \
    --subject_identifier_phrase '{subject_identifier_phrase}' \
    --output_directory {evaluation_output_directory}

## 10. Quick visual inspection

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

generated_image_paths = sorted(evaluation_output_directory.glob('gen_*.png'))
if not generated_image_paths:
    print('No generated images found yet; rerun Step 9.')
else:
    number_of_images_to_show = len(generated_image_paths)
    figure, axes_grid = plt.subplots(
        1,
        number_of_images_to_show,
        figsize=(3 * number_of_images_to_show, 3),
    )
    if number_of_images_to_show == 1:
        axes_grid = [axes_grid]
    for axis, generated_image_path in zip(axes_grid, generated_image_paths):
        axis.imshow(Image.open(generated_image_path))
        axis.set_title(generated_image_path.name, fontsize=8)
        axis.axis('off')
    plt.tight_layout()
    plt.show()

    clip_report_path = evaluation_output_directory / 'clip_report.txt'
    if clip_report_path.is_file():
        print(clip_report_path.read_text())